# calc_lethal(secondary=True) を使って追加効果込みの確定数を計算する

secondary引数の詳細（致死率計算に組み込まれている技に限り追加効果を加味する挙動）は
docs/api/README.md の calc_lethal() 節を参照。かえんほうしゃのやけど等、組み込まれて
いない技もある。キラースピン（どく付与）は組み込み済みで、secondary=Trueにすると
付与されたどくの毎ターンダメージ分だけ確定数が早まる。

Google Colabで開いた場合は、まず次のセルで `jpoke` をインストールする
（ローカルで `pip install jpoke` 済みなら不要）。

In [ ]:
%pip install -q jpoke

In [ ]:
from __future__ import annotations

from jpoke import Battle, Player

In [ ]:
poison_move = "キラースピン"
attacker_player = Player("SecondaryAttacker")
attacker_player.add_pokemon("ドクロッグ", move_names=[poison_move])
defender_player = Player("SecondaryDefender")
defender_player.add_pokemon("フシギダネ")

battle = Battle(attacker_player, defender_player, seed=1)
battle.start()
attacker = battle.get_active(attacker_player)

without_secondary = battle.calc_lethal(
    attacker=attacker, moves=poison_move, max_attack=8, secondary=False,
)[-1]
with_secondary = battle.calc_lethal(
    attacker=attacker, moves=poison_move, max_attack=8, secondary=True,
)[-1]
print(
    f"{poison_move}の確定数: secondary=False → {without_secondary.n_attack}発 / "
    f"secondary=True → {with_secondary.n_attack}発（どくの蓄積ダメージ分早まる）"
)

りゅうせいぐん等の「自身のとくこうを下げる」効果は secondary の対象外。

攻撃側自身に必ず発生する効果がsecondaryの指定に関わらず常に加味される理由は
docs/api/README.md の calc_lethal() 節を参照
（calc_lethal(secondary=False) でもとくこうの低下は反映される）。

In [ ]:
print("-" * 50)
move_name = "りゅうせいぐん"
attacker_player = Player("SelfDropAttacker")
attacker_player.add_pokemon("ボーマンダ", move_names=[move_name])
defender_player = Player("SelfDropDefender")
defender_player.add_pokemon("カビゴン")

battle = Battle(attacker_player, defender_player, seed=1)
battle.start()
attacker = battle.get_active(attacker_player)

without_secondary = battle.calc_lethal(
    attacker=attacker, moves=move_name, max_attack=2, secondary=False,
)
with_secondary = battle.calc_lethal(
    attacker=attacker, moves=move_name, max_attack=2, secondary=True,
)
print(
    f"{move_name}を2発撃った時点のとくこうランク: "
    f"secondary=False → {without_secondary[-1].attacker.boosts['spa']} / "
    f"secondary=True → {with_secondary[-1].attacker.boosts['spa']}（同じ値になる）"
)

試してみよう: 上の poison_move を「かえんほうしゃ」に変えると、
secondary=True でも確定数が変わらないこと（やけど付与が致死率計算に組み込まれて
いないこと）を確認できる